# 🎧 Generador Automático de Repasos en Audio (Maestro y Alumno) con Google AI Studio

Este cuaderno automatiza por completo la creación de tus repasos en audio del libro *Cumple completamente tu ministerio* (`pt14_S`):
1. **No tienes que copiar ni pegar las 529 partes a mano.**
2. **Conecta directamente al endpoint nativo `v1beta` de Google AI Studio** (compatible tanto con claves `AIza...` como con claves nuevas `AQ...`).
3. **Permite probar primero con 1 o 2 lecciones específicas** y **une automáticamente todas las partes** en un solo archivo `.mp3` listo para escuchar.


In [ ]:
# @title 1️⃣ (Opcional) Prepara el entorno y descarga los 529 guiones limpios { display-mode: "form" }
import subprocess, sys, urllib.request, zipfile
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", "google-genai>=2.24.0,<3"], check=True)

WORK_DIR = Path("/content/pt14_guiones")
PARTS_DIR = WORK_DIR / "03_Partes_Para_Google_AI_Studio"
ZIP_URL = "https://github.com/brandonrosaleslhest-del/jwpub-markdown-notebooklm/raw/main/pt14_S_Guiones_Audio_Maestro_Alumno.zip"

if not PARTS_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = WORK_DIR / "pt14_S_Guiones_Audio_Maestro_Alumno.zip"
    print("📥 Descargando tus 529 guiones limpios desde GitHub...")
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(WORK_DIR)

partes = sorted(f for f in PARTS_DIR.glob("[0-9]*.txt") if not f.name.startswith("00_"))
print(f"✅ ¡Listo! Se cargaron {len(partes)} partes limpias en {PARTS_DIR}")


In [ ]:
# @title 2️⃣ Elige las 1 o 2 lecciones de prueba, las voces y genera el audio unido (.mp3) { display-mode: "form" }
import base64, io, json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.request, wave, zipfile
from pathlib import Path

try:
    from google import genai
    from google.genai import types
except ImportError:
    print("📦 Instalando librería google-genai (solo la primera vez)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", "google-genai>=2.24.0,<3"], check=True)
    from google import genai
    from google.genai import types

from IPython.display import Audio, display
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = Path("/content/pt14_guiones")
PARTS_DIR = WORK_DIR / "03_Partes_Para_Google_AI_Studio"
ZIP_URL = "https://github.com/brandonrosaleslhest-del/jwpub-markdown-notebooklm/raw/main/pt14_S_Guiones_Audio_Maestro_Alumno.zip"
if not PARTS_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = WORK_DIR / "pt14_S_Guiones_Audio_Maestro_Alumno.zip"
    print("📥 Descargando tus 529 guiones limpios desde GitHub...")
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print("✅ Guiones listos.")

# @markdown ### 🔑 Pega aquí tu API Key de https://aistudio.google.com/apikey
GEMINI_API_KEY = "" # @param {type:"string"}

# @markdown ### 📚 Elige las lecciones que quieres generar ahora
# @markdown *(Para tu primera prueba rápida dejamos seleccionadas la **Lección 18 (1 parte)** y la **Lección 1A (4 partes)**)*
LECCION_1 = "35_Leccion_18_Palabras_finales (1 parte - Prueba rápida ~3 min)" # @param ["35_Leccion_18_Palabras_finales (1 parte - Prueba rápida ~3 min)", "36_Contraportada_Objetivos_de_la_Escuela_del_Servicio_de_P (1 parte - ~3 min)", "01_Leccion_1A_Que_te_espera_en_la_Escuela_del_Servicio_de (4 partes - ~18 min)", "02_Leccion_1B_Fortalece_tu_amistad_con_Jehova (10 partes)", "03_Leccion_2A_La_Traduccion_del_Nuevo_Mundo_parte_1 (6 partes)", "04_Leccion_2B_La_Traduccion_del_Nuevo_Mundo_parte_2 (9 partes)", "05_Leccion_3A_Mantente_al_dia (16 partes)", "06_Leccion_3B_Repaso_del_dia_1 (4 partes)", "07_Leccion_4A_La_soberania_de_Jehova_y_la_santificacion_de (16 partes)", "08_Leccion_4B_Interesate_sinceramente_por_los_demas (8 partes)", "09_Leccion_5A_La_predicacion_de_casa_en_casa_el_metodo_pri (20 partes)", "10_Leccion_5B_Taller_1_La_predicacion_de_casa_en_casa_el_m (3 partes)", "11_Leccion_6A_Mujeres_que_alegran_a_Jehova (25 partes)", "12_Leccion_6B_Repaso_del_dia_2 (7 partes)", "13_Leccion_7A_Como_te_ayudan_las_instrucciones_y_los_conse (17 partes)", "14_Leccion_7B_Lucha_contra_el_espiritu_del_mundo (24 partes)", "15_Leccion_8A_Lleva_una_vida_integra (21 partes)", "16_Leccion_8B_Participa_en_las_diferentes_formas_de_predic (18 partes)", "17_Leccion_9A_Taller_2_Participa_en_las_diferentes_formas (2 partes)", "18_Leccion_9B_Repaso_del_dia_3 (10 partes)", "19_Leccion_10A_Valora_el_papel_de_Jesus (29 partes)", "20_Leccion_10B_Piensa_en_principios (21 partes)", "21_Leccion_11A_Evalua_tu_progreso_como_persona_espiritual (34 partes)", "22_Leccion_11B_Haz_buenas_revisitas (15 partes)", "23_Leccion_12A_Taller_3_Haz_buenas_revisitas (3 partes)", "24_Leccion_12B_Repaso_del_dia_4 (10 partes)", "25_Leccion_13A_Aprende_del_Amo (27 partes)", "26_Leccion_13B_Da_cursos_biblicos_que_motiven_al_estudiant (17 partes)", "27_Leccion_14A_Da_cursos_biblicos_que_motiven_al_estudiant (15 partes)", "28_Leccion_14B_Taller_4_Da_cursos_biblicos_que_motiven_al (5 partes)", "29_Leccion_15A_Ayudalos_a_ser_cristianos_maduros (21 partes)", "30_Leccion_15B_Repaso_del_dia_5 (11 partes)", "31_Leccion_16A_La_felicidad_que_viene_de_Jehova_es_tu_fort (22 partes)", "32_Leccion_16B_Jehova_bendice_a_los_que_confian_en_el (26 partes)", "33_Leccion_17A_Nunca_dejes_de_orar (26 partes)", "34_Leccion_17B_Jehova_da_su_aprobacion_a_los_que_aguantan (25 partes)"]
LECCION_2 = "01_Leccion_1A_Que_te_espera_en_la_Escuela_del_Servicio_de (4 partes - ~18 min)" # @param ["Ninguna (solo generar Lección 1)", "01_Leccion_1A_Que_te_espera_en_la_Escuela_del_Servicio_de (4 partes - ~18 min)", "02_Leccion_1B_Fortalece_tu_amistad_con_Jehova (10 partes)", "03_Leccion_2A_La_Traduccion_del_Nuevo_Mundo_parte_1 (6 partes)", "04_Leccion_2B_La_Traduccion_del_Nuevo_Mundo_parte_2 (9 partes)", "05_Leccion_3A_Mantente_al_dia (16 partes)", "06_Leccion_3B_Repaso_del_dia_1 (4 partes)", "07_Leccion_4A_La_soberania_de_Jehova_y_la_santificacion_de (16 partes)", "08_Leccion_4B_Interesate_sinceramente_por_los_demas (8 partes)", "09_Leccion_5A_La_predicacion_de_casa_en_casa_el_metodo_pri (20 partes)", "10_Leccion_5B_Taller_1_La_predicacion_de_casa_en_casa_el_m (3 partes)", "11_Leccion_6A_Mujeres_que_alegran_a_Jehova (25 partes)", "12_Leccion_6B_Repaso_del_dia_2 (7 partes)", "13_Leccion_7A_Como_te_ayudan_las_instrucciones_y_los_conse (17 partes)", "14_Leccion_7B_Lucha_contra_el_espiritu_del_mundo (24 partes)", "15_Leccion_8A_Lleva_una_vida_integra (21 partes)", "16_Leccion_8B_Participa_en_las_diferentes_formas_de_predic (18 partes)", "17_Leccion_9A_Taller_2_Participa_en_las_diferentes_formas (2 partes)", "18_Leccion_9B_Repaso_del_dia_3 (10 partes)", "19_Leccion_10A_Valora_el_papel_de_Jesus (29 partes)", "20_Leccion_10B_Piensa_en_principios (21 partes)", "21_Leccion_11A_Evalua_tu_progreso_como_persona_espiritual (34 partes)", "22_Leccion_11B_Haz_buenas_revisitas (15 partes)", "23_Leccion_12A_Taller_3_Haz_buenas_revisitas (3 partes)", "24_Leccion_12B_Repaso_del_dia_4 (10 partes)", "25_Leccion_13A_Aprende_del_Amo (27 partes)", "26_Leccion_13B_Da_cursos_biblicos_que_motiven_al_estudiant (17 partes)", "27_Leccion_14A_Da_cursos_biblicos_que_motiven_al_estudiant (15 partes)", "28_Leccion_14B_Taller_4_Da_cursos_biblicos_que_motiven_al (5 partes)", "29_Leccion_15A_Ayudalos_a_ser_cristianos_maduros (21 partes)", "30_Leccion_15B_Repaso_del_dia_5 (11 partes)", "31_Leccion_16A_La_felicidad_que_viene_de_Jehova_es_tu_fort (22 partes)", "32_Leccion_16B_Jehova_bendice_a_los_que_confian_en_el (26 partes)", "33_Leccion_17A_Nunca_dejes_de_orar (26 partes)", "34_Leccion_17B_Jehova_da_su_aprobacion_a_los_que_aguantan (25 partes)", "35_Leccion_18_Palabras_finales (1 parte - Prueba rápida ~3 min)", "36_Contraportada_Objetivos_de_la_Escuela_del_Servicio_de_P (1 parte - ~3 min)"]

# @markdown ### 🎙️ Configura las dos voces y el modelo (deja "auto" para que detecte el compatible)
VOZ_MAESTRO = "Orus" # @param ["Orus", "Charon", "Schedar", "Sadaltager", "Sulafat", "Iapetus", "Algieba"]
VOZ_ALUMNO = "Puck" # @param ["Puck", "Fenrir", "Kore", "Aoede", "Callirrhoe", "Zephyr", "Achird"]
MODELO_TTS = "auto" # @param ["auto", "gemini-3.8-flash-tts", "gemini-3.8-flash-lite-tts", "gemini-3.1-flash-tts-preview", "gemini-2.5-flash-preview-tts", "gemini-2.5-pro-preview-tts"]
DESCARGAR_AUTOMATICAMENTE = True # @param {type:"boolean"}

STYLE_MAESTRO = (
    "Lee en español latinoamericano natural, con tono cálido, sereno, claro y didáctico, "
    "como un maestro cristiano experimentado explicando con cariño la Biblia."
)
STYLE_ALUMNO = (
    "Lee en español latinoamericano natural, con tono atento, respetuoso y reflexivo, "
    "como un estudiante aplicado haciendo las preguntas de la lección."
)

API_KEY = GEMINI_API_KEY.strip()
if not API_KEY:
    raise ValueError("⚠️ Por favor pega tu GEMINI_API_KEY en el cuadro de arriba (obtén una gratis en https://aistudio.google.com/apikey).")

# Forzamos vertexai=False y api_version="v1beta" para que las claves nuevas AQ... no se desvíen a Vertex Express
client = genai.Client(
    api_key=API_KEY,
    vertexai=False,
    http_options=types.HttpOptions(api_version="v1beta"),
)

CANDIDATE_MODELS = [
    "gemini-3.8-flash-tts",
    "gemini-3.8-flash-lite-tts",
    "gemini-3.1-flash-tts-preview",
    "gemini-2.5-flash-preview-tts",
    "gemini-2.5-pro-preview-tts",
    "gemini-2.5-flash-tts",
]

available_tts = []
try:
    req = urllib.request.Request(
        f"https://generativelanguage.googleapis.com/v1beta/models?pageSize=100",
        headers={"x-goog-api-key": API_KEY},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        for m in data.get("models", []):
            n = (m.get("name") or "").replace("models/", "")
            if "tts" in n.lower():
                available_tts.append(n)
except Exception as e:
    print(f"ℹ️ Aviso al consultar v1beta/models: {str(e)[:150]}")

models_order = []
if MODELO_TTS != "auto":
    models_order.append(MODELO_TTS)
for m in available_tts:
    if m not in models_order:
        models_order.append(m)
for m in CANDIDATE_MODELS:
    if m not in models_order:
        models_order.append(m)

print(f"🔎 Modelos TTS en generativelanguage v1beta: {available_tts or 'usando lista estándar'}")

def parse_turns(text):
    turns, cur_sp, cur_lines = [], None, []
    for raw in text.splitlines():
        ln = raw.strip()
        if not ln:
            if cur_lines:
                cur_lines.append("")
            continue
        if ln.startswith("Maestro:"):
            if cur_sp and any(cur_lines):
                turns.append((cur_sp, "\n".join(cur_lines).strip()))
            cur_sp, cur_lines = "Maestro", [ln[len("Maestro:"):].strip()]
        elif ln.startswith("Alumno:"):
            if cur_sp and any(cur_lines):
                turns.append((cur_sp, "\n".join(cur_lines).strip()))
            cur_sp, cur_lines = "Alumno", [ln[len("Alumno:"):].strip()]
        else:
            if cur_sp is None:
                cur_sp = "Maestro"
            cur_lines.append(ln)
    if cur_sp and any(cur_lines):
        turns.append((cur_sp, "\n".join(cur_lines).strip()))
    return [(s, t) for s, t in turns if t]

def extract_pcm(audio_bytes):
    if audio_bytes[:4] == b"RIFF":
        with wave.open(io.BytesIO(audio_bytes), "rb") as wf:
            return wf.readframes(wf.getnframes()), wf.getframerate(), wf.getnchannels(), wf.getsampwidth()
    return audio_bytes, 24000, 1, 2

def save_wav(path, pcm, rate=24000, channels=1, width=2):
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

def http_post_json(url, headers, payload):
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", **headers},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {e.code}: {body[:240]}") from e

def call_rest_interactions(model_id, turns, script_text):
    speakers = {s for s, _ in turns}
    if "3.8" in model_id:
        if len(speakers) == 1:
            sp = next(iter(speakers))
            v = VOZ_MAESTRO if sp == "Maestro" else VOZ_ALUMNO
            st = STYLE_MAESTRO if sp == "Maestro" else STYLE_ALUMNO
            payload = {
                "model": model_id,
                "input": [{"type": "text", "text": "\n\n".join(t for _, t in turns), "annotations": [{"type": "speech_metadata", "style": st}]}],
                "response_format": {"type": "audio"},
                "generation_config": {"speech_config": [{"voice": v}]},
            }
        else:
            payload = {
                "model": model_id,
                "input": [
                    {"type": "text", "text": t, "annotations": [{"type": "speech_metadata", "speaker": sp, "style": STYLE_MAESTRO if sp == "Maestro" else STYLE_ALUMNO}]}
                    for sp, t in turns
                ],
                "response_format": {"type": "audio"},
                "generation_config": {"speech_config": [{"speaker": "Maestro", "voice": VOZ_MAESTRO}, {"speaker": "Alumno", "voice": VOZ_ALUMNO}]},
            }
    else:
        if len(speakers) == 1:
            sp = next(iter(speakers))
            v = VOZ_MAESTRO if sp == "Maestro" else VOZ_ALUMNO
            st = STYLE_MAESTRO if sp == "Maestro" else STYLE_ALUMNO
            payload = {
                "model": model_id,
                "input": f"{st}\n\n" + "\n\n".join(t for _, t in turns),
                "response_format": {"type": "audio"},
                "generation_config": {"speech_config": [{"voice": v}]},
            }
        else:
            prompt = f"Lee en voz alta esta conversación de estudio bíblico en español natural.\nMaestro: {STYLE_MAESTRO}\nAlumno: {STYLE_ALUMNO}\n\n{script_text.strip()}"
            payload = {
                "model": model_id,
                "input": prompt,
                "response_format": {"type": "audio"},
                "generation_config": {"speech_config": [{"speaker": "Maestro", "voice": VOZ_MAESTRO}, {"speaker": "Alumno", "voice": VOZ_ALUMNO}]},
            }
    data = http_post_json(
        "https://generativelanguage.googleapis.com/v1beta/interactions",
        {"x-goog-api-key": API_KEY, "Api-Revision": "2026-05-20"},
        payload,
    )
    for step in reversed(data.get("steps", [])):
        for c in step.get("content", []):
            if c.get("type") == "audio" and c.get("data"):
                return base64.b64decode(c["data"])
    if "output_audio" in data and data["output_audio"].get("data"):
        return base64.b64decode(data["output_audio"]["data"])
    raise RuntimeError(f"Sin audio en respuesta: {str(data)[:180]}")

def call_rest_generate(model_id, turns, script_text):
    speakers = {s for s, _ in turns}
    if len(speakers) == 1:
        sp = next(iter(speakers))
        v = VOZ_MAESTRO if sp == "Maestro" else VOZ_ALUMNO
        st = STYLE_MAESTRO if sp == "Maestro" else STYLE_ALUMNO
        prompt = f"{st}\n\n" + "\n\n".join(t for _, t in turns)
        speech_cfg = {"voiceConfig": {"prebuiltVoiceConfig": {"voiceName": v}}}
    else:
        prompt = f"Lee en voz alta esta conversación de estudio bíblico en español natural.\nMaestro: {STYLE_MAESTRO}\nAlumno: {STYLE_ALUMNO}\n\n{script_text.strip()}"
        speech_cfg = {
            "multiSpeakerVoiceConfig": {
                "speakerVoiceConfigs": [
                    {"speaker": "Maestro", "voiceConfig": {"prebuiltVoiceConfig": {"voiceName": VOZ_MAESTRO}}},
                    {"speaker": "Alumno", "voiceConfig": {"prebuiltVoiceConfig": {"voiceName": VOZ_ALUMNO}}},
                ]
            }
        }
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseModalities": ["AUDIO"], "speechConfig": speech_cfg},
    }
    data = http_post_json(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent",
        {"x-goog-api-key": API_KEY},
        payload,
    )
    return base64.b64decode(data["candidates"][0]["content"]["parts"][0]["inlineData"]["data"])

def call_sdk_generate(model_id, turns, script_text):
    speakers = {s for s, _ in turns}
    if len(speakers) == 1:
        sp = next(iter(speakers))
        v = VOZ_MAESTRO if sp == "Maestro" else VOZ_ALUMNO
        st = STYLE_MAESTRO if sp == "Maestro" else STYLE_ALUMNO
        prompt = f"{st}\n\n" + "\n\n".join(t for _, t in turns)
        cfg = types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(voice_config=types.VoiceConfig(prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=v))),
        )
    else:
        prompt = f"Lee en voz alta esta conversación de estudio bíblico en español natural.\nMaestro: {STYLE_MAESTRO}\nAlumno: {STYLE_ALUMNO}\n\n{script_text.strip()}"
        cfg = types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(speaker="Maestro", voice_config=types.VoiceConfig(prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=VOZ_MAESTRO))),
                        types.SpeakerVoiceConfig(speaker="Alumno", voice_config=types.VoiceConfig(prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=VOZ_ALUMNO))),
                    ]
                )
            ),
        )
    resp = client.models.generate_content(model=model_id, contents=prompt, config=cfg)
    d = resp.candidates[0].content.parts[0].inline_data.data
    return base64.b64decode(d) if isinstance(d, str) else d

exhausted_models = set()

def synth_part(script_text, preferred_order):
    turns = parse_turns(script_text)
    last_err = None
    for m_id in preferred_order:
        if m_id in exhausted_models:
            continue
        model_daily_exhausted = False
        for m_name, fn in [
            ("rest_interactions", call_rest_interactions),
            ("rest_generate", call_rest_generate),
            ("sdk_generate", call_sdk_generate),
        ]:
            if model_daily_exhausted:
                break
            for retry in range(1, 4):
                try:
                    audio = fn(m_id, turns, script_text)
                    return audio, f"{m_id} [{m_name}]"
                except Exception as e:
                    last_err = e
                    msg = str(e)
                    low = msg.lower()
                    if any(k in low for k in ("per day", "requests per day", "rpd")):
                        print(f"   🔄 {m_id} llegó a su límite diario; cambiando al instante al siguiente modelo...")
                        exhausted_models.add(m_id)
                        model_daily_exhausted = True
                        break
                    if any(k in msg for k in ("404", "401", "403", "not found", "NOT_FOUND", "Unknown name", "INVALID_ARGUMENT", "400")):
                        print(f"   ↪️ {m_id} ({m_name}): {msg[:125]}")
                        break
                    wait_s = 15 * retry
                    print(f"   ⏳ {m_id} ({m_name}) ocupado: {msg[:110]}... esperando {wait_s}s")
                    time.sleep(wait_s)
    raise RuntimeError(
        f"Se agotó la cuota diaria en todos los modelos TTS disponibles por hoy. Último error: {last_err}\n"
        f"💡 Tus partes ya generadas siguen guardadas en caché; puedes continuar mañana o usar otra cuenta/proyecto de Google AI Studio."
    )

OUT_DIR = Path("/content/Audios_Escuela_Precursores")
CACHE_DIR = OUT_DIR / "partes_wav"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

chosen = [LECCION_1.split(" ")[0]]
if not LECCION_2.startswith("Ninguna"):
    k2 = LECCION_2.split(" ")[0]
    if k2 not in chosen:
        chosen.append(k2)

working_model = None
for lesson_prefix in chosen:
    pfiles = sorted(PARTS_DIR.glob(f"{lesson_prefix}_AIStudio*.txt"))
    print(f"\n============================================================")
    print(f"🎧 Generando {lesson_prefix} ({len(pfiles)} parte(s))")
    print(f"============================================================")
    combined_pcm = bytearray()
    rate, ch, width = 24000, 1, 2
    for i, pf in enumerate(pfiles, 1):
        c_wav = CACHE_DIR / f"{pf.stem}.wav"
        if c_wav.exists() and c_wav.stat().st_size > 1000:
            print(f"  [{i}/{len(pfiles)}] ✅ En caché: {pf.name}")
            raw = c_wav.read_bytes()
        else:
            print(f"  [{i}/{len(pfiles)}] 🎙️ Generando voz para {pf.name}...")
            order = [m for m in (([working_model] + [x for x in models_order if x != working_model]) if working_model else models_order) if m not in exhausted_models]
            raw, used_info = synth_part(pf.read_text(encoding="utf-8"), order)
            used_m = used_info.split(" ")[0]
            if working_model != used_m:
                working_model = used_m
                print(f"   ✅ Modelo activo: {used_info}")
            pcm_p, rate, ch, width = extract_pcm(raw)
            save_wav(c_wav, pcm_p, rate, ch, width)
            if i < len(pfiles):
                time.sleep(3)
        pcm_p, rate, ch, width = extract_pcm(raw)
        combined_pcm.extend(pcm_p)
        if i < len(pfiles):
            combined_pcm.extend(b"\x00" * int(rate * 0.6 * ch * width))

    final_wav = OUT_DIR / f"{lesson_prefix}_Maestro_Alumno.wav"
    final_mp3 = OUT_DIR / f"{lesson_prefix}_Maestro_Alumno.mp3"
    save_wav(final_wav, bytes(combined_pcm), rate, ch, width)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(final_wav), "-codec:a", "libmp3lame", "-qscale:a", "2", str(final_mp3)], check=False)
    target = final_mp3 if final_mp3.exists() else final_wav
    mins = (len(combined_pcm) / (rate * ch * width)) / 60.0
    print(f"🎉 ¡Terminado! {target.name} ({mins:.1f} min, {target.stat().st_size/1048576:.2f} MB)")
    display(Audio(str(target), autoplay=False))
    if IN_COLAB and DESCARGAR_AUTOMATICAMENTE:
        colab_files.download(str(target))
